# 01 - Exploratory Data Analysis

**IBM HR Analytics Employee Attrition & Performance**

The dataset is fictional (generated by IBM data scientists, released under CC0), so it
carries no real personal data. Every conclusion drawn here is a statement about the
dataset, not about real employment decisions.

Run this notebook from the repository root so `ml.common` is importable:
```
jupyter notebook notebooks/01_EDA.ipynb
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "ml" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.common import features as F
from ml.common import io

FIG_DIR = ROOT / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Carbon Design System palette - see docs/design.md
CARBON = {
    "blue": "#0f62fe", "purple": "#6929c4", "teal": "#009d9a",
    "cyan": "#1192e8", "magenta": "#9f1853", "red": "#da1e28",
    "green": "#198038", "orange": "#ff832b", "gray": "#8d8d8d",
    "text": "#161616", "muted": "#525252", "grid": "#e0e0e0",
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.edgecolor": CARBON["grid"],
    "axes.labelcolor": CARBON["muted"],
    "xtick.color": CARBON["muted"],
    "ytick.color": CARBON["muted"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

path = io.download_ibm_hr()
df = F.load_dataset(path)
print(f"{path.name}  sha256={io.sha256(path)[:16]}...")
print(f"shape: {df.shape}")

## 1. Shape and headline distributions

In [ ]:
attrition_rate = (df["Attrition"] == "Yes").mean()
print(f"Attrition rate: {attrition_rate:.1%}  ({(df['Attrition'] == 'Yes').sum()} of {len(df)})")
print(f"Departments: {df['Department'].value_counts().to_dict()}")
print(f"Gender:      {df['Gender'].value_counts().to_dict()}")
print(f"Age range:   {df['Age'].min()}-{df['Age'].max()},  median {df['Age'].median():.0f}")
print(f"Tenure:      median {df['YearsAtCompany'].median():.0f} years, max {df['YearsAtCompany'].max()}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].bar(df["Department"].value_counts().index, df["Department"].value_counts().values, color=CARBON["blue"])
axes[0].set_title("Headcount by department")
axes[0].tick_params(axis="x", rotation=12)

axes[1].hist(df["Age"], bins=20, color=CARBON["purple"])
axes[1].set_title("Age distribution")

axes[2].hist(df["MonthlyIncome"], bins=40, color=CARBON["teal"])
axes[2].set_title("Monthly income")

for ax in axes:
    ax.grid(axis="y", color=CARBON["grid"])
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIG_DIR / "01_overview.png", dpi=150)
plt.show()

## 2. Attrition rate by feature

This is the exploratory view that motivates the model. Overtime and job satisfaction
are the two clearest separations.

In [ ]:
def rate_by(col, order=None):
    g = df.groupby(col)["Attrition"].apply(lambda s: (s == "Yes").mean()).sort_values(ascending=False)
    if order is not None:
        g = g.reindex(order)
    return g

panels = [
    ("OverTime", rate_by("OverTime"), CARBON["red"]),
    ("JobSatisfaction", rate_by("JobSatisfaction", [1, 2, 3, 4]), CARBON["blue"]),
    ("JobLevel", rate_by("JobLevel"), CARBON["purple"]),
    ("Department", rate_by("Department"), CARBON["teal"]),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (title, series, color) in zip(axes, panels):
    ax.bar([str(i) for i in series.index], series.values, color=color)
    ax.set_title(f"Attrition rate by {title}")
    ax.grid(axis="y", color=CARBON["grid"])
    ax.set_axisbelow(True)
    ax.set_ylim(0, max(series.max() * 1.25, 0.05))
    for x, v in enumerate(series.values):
        ax.text(x, v + 0.005, f"{v:.0%}", ha="center", fontsize=9, color=CARBON["muted"])

plt.tight_layout()
plt.savefig(FIG_DIR / "02_attrition_by_feature.png", dpi=150)
plt.show()

## 3. Protected attributes - where the model could be unfair

The baseline rates below are *not* evidence of model unfairness; they are the prior
the model starts from. The audit in `ml/train_fairness.py` measures the model's
behaviour against them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

for ax, col, color in zip(axes, ["Gender", "AgeBand", "MaritalStatus"], [CARBON["blue"], CARBON["purple"], CARBON["teal"]]):
    series = df.groupby(col)["Attrition"].apply(lambda s: (s == "Yes").mean())
    ax.bar([str(i) for i in series.index], series.values, color=color)
    ax.set_title(f"Attrition rate by {col}")
    ax.grid(axis="y", color=CARBON["grid"])
    ax.set_axisbelow(True)
    for x, v in enumerate(series.values):
        ax.text(x, v + 0.005, f"{v:.0%}", ha="center", fontsize=9, color=CARBON["muted"])

plt.tight_layout()
plt.savefig(FIG_DIR / "03_protected_attributes.png", dpi=150)
plt.show()

## 4. Leakage check on the performance rating

This is the single most important piece of analysis in the project, and it belongs in
the exploratory phase rather than the modelling phase - because it changes what the
project is allowed to build.

`PerformanceRating` is a **deterministic function of `PercentSalaryHike`**: every
employee with the same raise percentage carries the same rating. A model with the
raise as a feature scores a perfect RMSE of 0.0000 and an R-squared of 1.0000, which
measures nothing.

With the proxy removed, the rating becomes *unpredictable* - cross-validated ROC-AUC
of 0.457 for distinguishing a rating of 4 from 3, which is below chance.

So the talent matrix is built from **observed** engagement and growth indicators
instead of a predicted rating. See `docs/model_cards/ninebox.md`.

In [ ]:
cleaned = F.clean(df)

groups = cleaned.groupby("PercentSalaryHike")["PerformanceRating"].nunique()
print(f"Distinct PerformanceRating values per PercentSalaryHike level: max = {groups.max()}")
print("-> 1 means the rating is a deterministic function of the salary hike.\n")

# Note: use the uncleaned frame here - F.clean() drops the EmployeeNumber identifier.
pivot = df.pivot_table(
    index="PercentSalaryHike", columns="PerformanceRating", values="EmployeeNumber", aggfunc="count", fill_value=0
)
print(pivot.to_string())

fig, ax = plt.subplots(figsize=(7, 3.6))
for col, color in zip(pivot.columns, [CARBON["orange"], CARBON["blue"]]):
    ax.plot(pivot.index, pivot[col], marker="o", label=f"PerformanceRating = {col}", color=color)
ax.set_xlabel("PercentSalaryHike")
ax.set_ylabel("Employees")
ax.set_title("Rating is fully determined by the salary hike")
ax.legend(frameon=False)
ax.grid(axis="y", color=CARBON["grid"])
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_rating_leakage.png", dpi=150)
plt.show()

## 5. Correlation overview

Note how weak the numeric correlations are overall. This dataset has only one
genuinely predictive target, which is why the project is careful about what it claims
to predict.

In [ ]:
numeric = cleaned.select_dtypes("number")
target = (cleaned["Attrition"] == "Yes").astype(int)
corr = numeric.corrwith(target).sort_values(key=abs, ascending=False)
print("Correlation with Attrition (numeric features only):")
print(corr.head(12).round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 4.2))
top = corr.head(12)
colors = [CARBON["red"] if v > 0 else CARBON["blue"] for v in top]
ax.barh([str(i) for i in top.index][::-1], top.values[::-1], color=colors[::-1])
ax.set_xlabel("Pearson correlation with Attrition")
ax.set_title("Numeric features vs attrition")
ax.grid(axis="x", color=CARBON["grid"])
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_correlations.png", dpi=150)
plt.show()

## Next

- `notebooks/02_model_experiments.ipynb` - model comparison, calibration study, threshold tuning
- `notebooks/03_fairness_analysis.ipynb` - the audit, the mitigations, and the detector power curve

Both are produced by the scripts in `ml/`; the scripts are the source of truth and the
notebooks are the readable narrative.